### Single forward for explainability

One MRI, one prompt, one pass through Stage B — each block called by hand, in the order the data meets it.

Three things reach the model: the image, three anchor **names**, three **direction ids**. Everything else is derived inside. The mask path is for scoring only; it never enters `forward`.


In [21]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").is_file())
sys.path.insert(0, str(ROOT))
OUT = ROOT / "vizualization" / "out"
OUT.mkdir(parents=True, exist_ok=True)

from src.config import load_config
from src.data import Corpus, load_nifti, normalize
from src.engine import dice_iou, load_model, load_stage_a, mask_centroid_world, resolve_device
from src.geometry import centroids_world, solutions_for, volume_center_world
from src.models import LOG_FLOOR, StageB, soft_argmax


def resolve(path: str | Path) -> Path:
    path = Path(path)
    return path if path.is_absolute() else ROOT / path


def shown(path: Path) -> str:
    try:
        return str(path.resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return str(path)


def describe(name: str, tensor: torch.Tensor) -> None:
    t = tensor.detach()
    print(
        f"  {name:<22} {str(tuple(t.shape)):<28} {str(t.dtype):<12} "
        f"min={t.min().item():.4g}  max={t.max().item():.4g}  mean={t.float().mean().item():.4g}"
    )


def world_to_index(xyz, shape, spacing):
    """World (x, y, z) -> voxel (z, y, x), clipped to the volume."""
    ix = int(np.clip(round(float(xyz[0]) / spacing[0]), 0, shape[2] - 1))
    iy = int(np.clip(round(float(xyz[1]) / spacing[1]), 0, shape[1] - 1))
    iz = int(np.clip(round(float(xyz[2]) / spacing[2]), 0, shape[0] - 1))
    return iz, iy, ix


def grey(plane: np.ndarray) -> go.Heatmap:
    lo, hi = np.percentile(plane, (1, 99))
    if not np.isfinite(lo) or lo == hi:
        lo, hi = float(np.nanmin(plane)), float(np.nanmax(plane) + 1e-6)
    return go.Heatmap(z=plane, colorscale="gray", zmin=lo, zmax=hi, showscale=False, hoverinfo="skip")


def overlay(plane: np.ndarray, colour: str, name: str, threshold: float = 0.05) -> go.Heatmap:
    return go.Heatmap(
        z=np.where(plane > threshold, plane, np.nan),
        colorscale=[[0, "rgba(0,0,0,0)"], [1, colour]],
        zmin=0, zmax=1, showscale=False, name=name,
        hovertemplate=f"{name}: %{{z:.3f}}<extra></extra>",
    )


def contour(plane: np.ndarray, colour: str, name: str) -> go.Contour:
    return go.Contour(
        z=plane, contours=dict(start=0.5, end=0.5, size=0, coloring="none"),
        line=dict(color=colour, width=2), showscale=False, name=name, hoverinfo="skip",
    )


def show(fig: go.Figure, stem: str = "forward", panel: int = 340, **_kw) -> None:
    """Write HTML and display it. Each subplot is square, matching the cubic input slices."""
    n_rows = max(len({round(fig.layout[k].domain[0], 4) for k in fig.layout if k.startswith("yaxis")}), 1)
    n_cols = max(len({round(fig.layout[k].domain[0], 4) for k in fig.layout if k.startswith("xaxis")}), 1)
    margin = dict(t=90, b=36, l=36, r=56)
    fig.update_layout(
        autosize=False,
        width=margin["l"] + margin["r"] + n_cols * panel,
        height=margin["t"] + margin["b"] + n_rows * panel,
        template="plotly_dark",
        margin=margin,
    )
    # Each y-axis must lock to *its* x-axis. `scaleanchor="x"` on every y-axis
    # pins later panels to the first one and stretches them.
    fig.for_each_yaxis(lambda axis: axis.update(
        autorange="reversed", scaleanchor=axis.anchor, scaleratio=1,
        constrain="domain", showticklabels=False, showgrid=False, zeroline=False,
    ))
    fig.for_each_xaxis(lambda axis: axis.update(
        constrain="domain", showticklabels=False, showgrid=False, zeroline=False,
        scaleratio=1,
    ))
    path = OUT / f"{stem}.html"
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    print(f"  -> {shown(path)}")
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


##### 00. Define inputs

Checkpoint paths are required. Leave `image_path` / `prompt` blank to take the first val example of the corpus. `mask_path` is a corpus label volume or a binary target mask — scoring only, never an argument to the model. Leave `stage_a_path` blank to use the Stage A frozen inside the Stage B checkpoint.


In [22]:
image_path = "data/synthetic-hard/scenes/test_00000/image.nii.gz"
mask_path = "data/synthetic-hard/scenes/test_00000/labels.nii.gz"
prompt = "segment the structure that is lateral to the triangular_prism, superior to the cube, and anterior to the cylinder."

stage_a_path = "runs/hard-stage-a/best.pt"   # empty -> Stage A frozen inside Stage B
stage_b_path = "runs/hard-stage-b/best.pt"
config_path = "configs/synthetic-hard.yaml"
device_name = "auto"


##### 1. Load the models

Stage B's checkpoint is self-contained (it carries the frozen Stage A it was trained with). A separate Stage A path, when given, replaces that segmenter so both files are the ones that run.


In [23]:
cfg = load_config(resolve(config_path))
corpus = Corpus.load(resolve(cfg.data.root))
vocab = corpus.vocab
device = resolve_device(device_name)

stage_b_file = resolve(stage_b_path)
if not stage_b_file.is_file():
    raise FileNotFoundError(f"Stage B checkpoint not found: {stage_b_file}")

model = load_model(stage_b_file, device)
if not isinstance(model, StageB):
    raise TypeError(f"expected a Stage B checkpoint, got {type(model).__name__} from {stage_b_file}")

if str(stage_a_path).strip():
    stage_a_file = resolve(stage_a_path)
    if not stage_a_file.is_file():
        raise FileNotFoundError(f"Stage A checkpoint not found: {stage_a_file}")
    segmenter = load_stage_a(stage_a_file, device)
    model.load_segmenter(segmenter)
    print(f"Stage A  {shown(stage_a_file)}  (injected into Stage B)")
else:
    segmenter = model.segmenter
    print("Stage A  (frozen copy inside the Stage B checkpoint)")

model.eval()
segmenter.eval()
spacing = tuple(float(v) for v in model.spacing)
threshold = float(cfg.train.threshold)

print(f"Stage B  {shown(stage_b_file)}")
print(f"device   {device}  ·  resolution {model.resolution}³  ·  spacing {spacing}"
      f"  ·  tau={model.mapper.tau}  min_mass={model.mapper.min_mass:.1e}")
print(f"vocab    {len(vocab)} names from {shown(corpus.root)}")
if tuple(float(v) for v in corpus.spacing) != spacing:
    print(f"warning: corpus spacing {corpus.spacing} != checkpoint spacing {spacing}; "
          "the mapper uses the checkpoint")


Stage A  /home/imag2/Documents/IMAG2/SpatialVox-queryhead/runs/hard-stage-a/best.pt  (injected into Stage B)
Stage B  /home/imag2/Documents/IMAG2/SpatialVox-queryhead/runs/hard-stage-b/best.pt
device   cuda  ·  resolution 64³  ·  spacing (1.0, 1.0, 1.0)  ·  tau=0.5  min_mass=1.0e-06
vocab    10 names from /home/imag2/Documents/IMAG2/SpatialVox-queryhead/data/synthetic-hard


In [24]:
def first_corpus_example():
    for split in ("val", "train", "test"):
        try:
            rows = corpus.records(split)
        except FileNotFoundError:
            continue
        if not rows:
            continue
        row = rows[0]
        scene = corpus.root / "scenes" / row["scene"]
        return scene / "image.nii.gz", scene / "labels.nii.gz", row["prompt"], row
    raise FileNotFoundError(f"no examples in {corpus.root}")


if not str(image_path).strip():
    image_file, default_mask, default_prompt, row = first_corpus_example()
    prompt_text = prompt.strip() if str(prompt).strip() else default_prompt
    mask_file = resolve(mask_path) if str(mask_path).strip() else default_mask
    print(f"using corpus example {row['id']}  target={vocab.name(row['target'])}")
else:
    if not str(prompt).strip():
        raise ValueError("prompt is required when image_path is set")
    image_file = resolve(image_path)
    mask_file = resolve(mask_path) if str(mask_path).strip() else None
    prompt_text = prompt
    row = None

image_np = normalize(load_nifti(image_file, np.float32), cfg.data.normalize)
expected = (model.resolution,) * 3
if image_np.shape != expected:
    raise ValueError(f"image shape {image_np.shape} != {expected} this checkpoint was built for")

header_spacing = tuple(float(v) for v in nib.load(str(image_file)).header.get_zooms()[:3])
print(f"image    {shown(image_file)}")
print(f"         shape {image_np.shape} (z,y,x)  nifti zooms {header_spacing}  normalize={cfg.data.normalize}")

clauses = vocab.parse(prompt_text)
direction_ids_list, name_ids_list = vocab.clause_ids(clauses)
print(f"prompt   {prompt_text}")
for i, clause in enumerate(clauses):
    print(f"  slot {i}: the target is {clause['direction']:<9} of {clause['anchor']}  "
          f"(dir_id={direction_ids_list[i]}, name_id={name_ids_list[i]})")

# Scoring only. A label volume is solved with the corpus rule; a binary mask is used as-is.
labels_np = None
target_mask = None
target_name = None
truth_c = None
if mask_file is not None and Path(mask_file).is_file():
    labels_np = load_nifti(mask_file, np.int32)
    if labels_np.max() <= 1:
        target_mask = (labels_np > 0).astype(np.float32)
        target_name = "(binary mask)"
        print(f"mask     {shown(Path(mask_file))}  binary, {int(target_mask.sum())} voxels")
    else:
        present = [int(v) for v in np.unique(labels_np) if v != 0]
        centroids = centroids_world(labels_np, len(vocab), spacing)
        center = volume_center_world(labels_np.shape, spacing)
        anchor_labels = [vocab.label(c["anchor"]) for c in clauses]
        hits = solutions_for(anchor_labels, [c["direction"] for c in clauses], centroids, present, center)
        if len(hits) == 1:
            target_mask = (labels_np == hits[0]).astype(np.float32)
            target_name = vocab.name(hits[0])
            print(f"mask     {shown(Path(mask_file))}  label volume → {target_name}  ({int(target_mask.sum())} voxels)")
        else:
            print(f"mask     {shown(Path(mask_file))}  conjunction names {len(hits)} structure(s); no target to score")
elif mask_file is not None:
    print(f"mask     {shown(Path(mask_file))}  (not found, scoring skipped)")

if target_mask is not None and target_mask.any():
    truth_c, _ = mask_centroid_world(torch.from_numpy(target_mask)[None, None], spacing)

image = torch.from_numpy(np.ascontiguousarray(image_np))[None, None].to(device)
direction_ids = torch.tensor([direction_ids_list], dtype=torch.long, device=device)
name_ids = torch.tensor([name_ids_list], dtype=torch.long, device=device)
print(f"tensors  image {tuple(image.shape)}  direction_ids {tuple(direction_ids.shape)}  name_ids {tuple(name_ids.shape)}")


image    /home/imag2/Documents/IMAG2/SpatialVox-queryhead/data/synthetic-hard/scenes/test_00000/image.nii.gz
         shape (64, 64, 64) (z,y,x)  nifti zooms (1.0, 1.0, 1.0)  normalize=none
prompt   segment the structure that is lateral to the triangular_prism, superior to the cube, and anterior to the cylinder.
  slot 0: the target is lateral   of triangular_prism  (dir_id=5, name_id=7)
  slot 1: the target is superior  of cube  (dir_id=2, name_id=0)
  slot 2: the target is anterior  of cylinder  (dir_id=0, name_id=4)
mask     /home/imag2/Documents/IMAG2/SpatialVox-queryhead/data/synthetic-hard/scenes/test_00000/labels.nii.gz  label volume → pyramid  (2005 voxels)
tensors  image (1, 1, 64, 64, 64)  direction_ids (1, 3)  name_ids (1, 3)


##### 2. Obtain anchors

`A_i = stop_gradient(sigmoid(stage_a(image, name_ids)_i))`. Soft, detached, and the only thing a name ever buys. Names stop here.


In [25]:
ANCHOR_COLOURS = ("#e64a19", "#1e88e5", "#43a047")
TARGET_COLOUR = "#ffd600"
WHERE_COLOUR = "#ff6f00"
PRED_COLOUR = "#00e5ff"

with torch.no_grad():
    anchors = segmenter.probability(image, name_ids).detach()   # [1, 3, D, H, W]

print("anchors  A_i = stop_gradient(sigmoid(Stage A))")
describe("A", anchors)
print(f"  {'slot':<5}{'structure':<24}{'mass':>11}{'max p':>8}  soft centroid (x,y,z)")
slot_z = []
for slot, clause in enumerate(clauses):
    mass = float(anchors[0, slot].mean())
    total = float(anchors[0, slot].sum())
    depth, height, width = anchors.shape[2:]
    zz = torch.arange(depth, device=anchors.device, dtype=torch.float32)
    yy = torch.arange(height, device=anchors.device, dtype=torch.float32)
    xx = torch.arange(width, device=anchors.device, dtype=torch.float32)
    cx = float((anchors[0, slot].sum(dim=(0, 1)) * xx).sum() / (total + 1e-8)) * spacing[0]
    cy = float((anchors[0, slot].sum(dim=(0, 2)) * yy).sum() / (total + 1e-8)) * spacing[1]
    cz = float((anchors[0, slot].sum(dim=(1, 2)) * zz).sum() / (total + 1e-8)) * spacing[2]
    z_slice = int(np.clip(round(cz / spacing[2]), 0, depth - 1))
    slot_z.append(z_slice)
    flag = "  <- below min_mass, F_i will be 0" if mass < model.mapper.min_mass else ""
    print(f"  {slot:<5}{clause['anchor']:<24}{mass:>11.2e}{float(anchors[0, slot].max()):>8.3f}  "
          f"({cx:6.1f},{cy:6.1f},{cz:6.1f}){flag}")

img = image_np
fig = make_subplots(
    rows=1, cols=3, horizontal_spacing=0.04,
    subplot_titles=[
        f"slot {i}: {c['anchor']}<br><sub>{c['direction']} → the target · z={z}</sub>"
        for i, (c, z) in enumerate(zip(clauses, slot_z))
    ],
)
for slot, z in enumerate(slot_z):
    fig.add_trace(grey(img[z]), row=1, col=slot + 1)
    fig.add_trace(overlay(anchors[0, slot, z].cpu().numpy(), ANCHOR_COLOURS[slot], f"A_{slot}"), row=1, col=slot + 1)
    if target_mask is not None:
        fig.add_trace(contour(target_mask[z], TARGET_COLOUR, "target"), row=1, col=slot + 1)
fig.update_layout(title="2 · A_i — detached soft anchor masks, each through its own centroid")
show(fig, stem="forward_2_anchors")


anchors  A_i = stop_gradient(sigmoid(Stage A))
  A                      (1, 3, 64, 64, 64)           torch.float32 min=1.557e-06  max=0.9997  mean=0.01125
  slot structure                      mass   max p  soft centroid (x,y,z)
  0    triangular_prism           9.81e-03   1.000  (  39.1,  53.5,  35.4)
  1    cube                       1.74e-02   0.996  (  28.9,  43.5,  29.4)
  2    cylinder                   6.48e-03   1.000  (  13.6,  22.9,  43.1)
  -> vizualization/out/forward_2_anchors.html


##### 3. Positional mapper

No parameters. It never sees the image or a name. The direction id is consumed here and nowhere else.

`F_i = sigmoid(margin_i / τ)` if `mass_i ≥ min_mass`, else 0. `where_raw = F_0 · F_1 · F_2`, never renormalised.


In [26]:
with torch.no_grad():
    field = model.mapper(anchors, direction_ids, model.spacing, model.center)

print("mapper   PositionalMapper3D  (zero parameters)")
describe("F", field.fields)
describe("where_raw", field.where_raw)
describe("where_mass", field.where_mass)
describe("masses", field.masses)
print(f"  where_mass = {float(field.where_mass):.3e}   (the null head's first number)")
print(f"  volume fraction where_raw > 0.05 = {float((field.where_raw > 0.05).float().mean()):.2%}")

z_target = image_np.shape[0] // 2
if truth_c is not None:
    z_target, y_target, x_target = world_to_index(truth_c[0], image_np.shape, spacing)
    at = float(field.where_raw[0, 0, z_target, y_target, x_target])
    inside = float(
        (torch.from_numpy(target_mask) * (field.where_raw[0, 0].cpu() > 0.05)).sum()
        / max(float(target_mask.sum()), 1.0)
    )
    print(f"  where_raw at the target centroid = {at:.4f}   (gate asks: > 0.5?)")
    print(f"  target voxels inside where_raw > 0.05 = {inside:.1%}")

fields_np = field.fields[0].cpu().numpy()
where_np = field.where_raw[0, 0].cpu().numpy()
fig = make_subplots(
    rows=1, cols=4, horizontal_spacing=0.03,
    subplot_titles=[f"F_{i}: {c['direction']}<br><sub>of {c['anchor']}</sub>" for i, c in enumerate(clauses)]
                   + ["<b>where_raw = F₀·F₁·F₂</b><br><sub>never renormalised</sub>"],
)
for slot in range(3):
    fig.add_trace(grey(img[z_target]), row=1, col=slot + 1)
    fig.add_trace(overlay(fields_np[slot][z_target], ANCHOR_COLOURS[slot], f"F_{slot}", 0.02), row=1, col=slot + 1)
fig.add_trace(grey(img[z_target]), row=1, col=4)
fig.add_trace(overlay(where_np[z_target], WHERE_COLOUR, "where_raw", 0.02), row=1, col=4)
if target_mask is not None:
    fig.add_trace(contour(target_mask[z_target], TARGET_COLOUR, "target"), row=1, col=4)
fig.update_layout(title=f"3 · mapper — three pyramids and their product  ·  axial z={z_target}")
show(fig, stem="forward_3_mapper", width=1400)


mapper   PositionalMapper3D  (zero parameters)
  F                      (1, 3, 64, 64, 64)           torch.float32 min=0  max=1  mean=0.214
  where_raw              (1, 1, 64, 64, 64)           torch.float32 min=0  max=1  mean=0.0333
  where_mass             (1, 1)                       torch.float32 min=0.0333  max=0.0333  mean=0.0333
  masses                 (1, 3)                       torch.float32 min=0.006481  max=0.01745  mean=0.01125
  where_mass = 3.330e-02   (the null head's first number)
  volume fraction where_raw > 0.05 = 4.33%
  where_raw at the target centroid = 0.0000   (gate asks: > 0.5?)
  target voxels inside where_raw > 0.05 = 12.3%
  -> vizualization/out/forward_3_mapper.html


##### 4. Boundary encoder

`B(I)`: generic boundary features from the MRI, and nothing else. No prompt, no name, no direction, no coordinate grid.


In [27]:
with torch.no_grad():
    if model.boundary is None:
        boundary = None
        print("B(I)     this checkpoint is the prompt-only ablation; the carver sees geometry alone")
    else:
        boundary = model.boundary(image.to(torch.float32))
        print("B(I)     BoundaryEncoder  (image only)")
        describe("B(I)", boundary)

if boundary is not None:
    feat = boundary[0].float().cpu().numpy()
    depth_frac = z_target / max(image_np.shape[0] - 1, 1)
    z_b = int(np.clip(round(depth_frac * (feat.shape[1] - 1)), 0, feat.shape[1] - 1))
    busiest = int(feat.reshape(feat.shape[0], -1).var(1).argmax())
    fig = make_subplots(
        rows=1, cols=2, horizontal_spacing=0.06,
        subplot_titles=["mean over channels", f"highest-variance channel ({busiest})"],
    )
    fig.add_trace(
        go.Heatmap(z=feat.mean(0)[z_b], colorscale="Viridis", showscale=True,
                   colorbar=dict(len=0.8, x=0.46, thickness=12)),
        row=1, col=1,
    )
    fig.add_trace(
        go.Heatmap(z=feat[busiest][z_b], colorscale="Inferno", showscale=True,
                   colorbar=dict(len=0.8, x=1.02, thickness=12)),
        row=1, col=2,
    )
    fig.update_layout(title=f"4 · B(I)  ·  {tuple(boundary.shape[1:])}  ·  relative depth z≈{depth_frac:.0%}")
    show(fig, stem="forward_4_boundary", width=900)


B(I)     BoundaryEncoder  (image only)
  B(I)                   (1, 16, 64, 64, 64)          torch.float32 min=-0.08006  max=10.09  mean=0.3699
  -> vizualization/out/forward_4_boundary.html


##### 5. Null head

`valid = MLP(where_mass, mass_0, mass_1, mass_2)`. Four scalars, no pixels — it cannot decide "empty" by looking at tissue. Logit > 0 means the clauses name something.


In [28]:
with torch.no_grad():
    valid = model.null(field.where_mass, field.masses)

print("null     MLP on log10(where_mass, mass_0, mass_1, mass_2)")
describe("valid", valid)
print(f"  logit = {float(valid):+.4f}   →  {'names something' if float(valid) > 0 else 'names nothing'}")
print(f"  inputs (raw):  where_mass={float(field.where_mass):.3e}  "
      f"masses={[f'{float(m):.3e}' for m in field.masses[0]]}")


null     MLP on log10(where_mass, mass_0, mass_1, mass_2)
  valid                  (1,)                         torch.float32 min=3.184  max=3.184  mean=3.184
  logit = +3.1840   →  names something
  inputs (raw):  where_mass=3.330e-02  masses=['9.814e-03', '1.745e-02', '6.481e-03']


##### 6. Carver

The WHAT. Concatenation in is `B(I)`, the three soft masks, the three fields, `where_raw`, `log(where_raw)`, `where_mass`. No name, no direction id, no coordinate grid. Anchor voxels are then written to the background logit — they are the given, not the answer.


In [29]:
where = field.where_raw
log_where = where.clamp_min(LOG_FLOOR).log() / -math.log(LOG_FLOOR)
log_mass = (
    field.where_mass.clamp_min(LOG_FLOOR).log() / -math.log(LOG_FLOOR)
).reshape(-1, 1, 1, 1, 1).expand_as(where)

parts = [anchors, field.fields, where, log_where, log_mass]
dtype = boundary.dtype if boundary is not None else torch.float32
carver_in = torch.cat(([boundary] if boundary is not None else []) + [p.to(dtype) for p in parts], dim=1)

print("carver   channels:")
print(f"  {'B(I)':<22} {tuple(boundary.shape) if boundary is not None else '(absent)'}")
print(f"  {'A_0..2':<22} {tuple(anchors.shape)}")
print(f"  {'F_0..2':<22} {tuple(field.fields.shape)}")
print(f"  {'where_raw':<22} {tuple(where.shape)}")
print(f"  {'log(where_raw)':<22} {tuple(log_where.shape)}")
print(f"  {'where_mass (broadcast)':<22} {tuple(log_mass.shape)}")
describe("carver_in", carver_in)

with torch.no_grad():
    logits, heatmap = model.carver(carver_in, boundary)
    if model.alpha is not None:
        logits = logits + model.alpha * torch.logit(where.clamp(1e-4, 1 - 1e-4))
    # predicted soft mask, never the label volume
    logits = logits.masked_fill(anchors.amax(dim=1, keepdim=True) > 0.5, model.background_logit)
    centroid = soft_argmax(heatmap, (model.resolution,) * 3, model.spacing)

describe("logits", logits)
describe("heatmap", heatmap)
print(f"  predicted centroid (x,y,z) = {tuple(round(float(v), 2) for v in centroid[0])}")


carver   channels:
  B(I)                   (1, 16, 64, 64, 64)
  A_0..2                 (1, 3, 64, 64, 64)
  F_0..2                 (1, 3, 64, 64, 64)
  where_raw              (1, 1, 64, 64, 64)
  log(where_raw)         (1, 1, 64, 64, 64)
  where_mass (broadcast) (1, 1, 64, 64, 64)
  carver_in              (1, 25, 64, 64, 64)          torch.float32 min=-1  max=10.09  mean=0.2216
  logits                 (1, 1, 64, 64, 64)           torch.float32 min=-51.82  max=11.39  mean=-14.25
  heatmap                (1, 1, 32, 32, 32)           torch.float32 min=-9.956  max=11.98  mean=-1.358
  predicted centroid (x,y,z) = (17.82, 51.01, 35.86)


##### 7. Outputs

The mask from the logits, the centroid from the heatmap (not from the mask), the null logit. A Dice on one example is illustrative — not a result.


In [30]:
probability = torch.sigmoid(logits[0, 0].float()).cpu().numpy()
predicted = probability >= threshold
z = z_target

fig = make_subplots(
    rows=1, cols=3, horizontal_spacing=0.04,
    subplot_titles=[
        "predicted probability",
        "thresholded vs truth" if target_mask is not None else "thresholded mask",
        "heatmap logits<br><sub>soft-argmax integrates their softmax</sub>",
    ],
)
fig.add_trace(grey(img[z]), row=1, col=1)
fig.add_trace(overlay(probability[z], PRED_COLOUR, "p", 0.02), row=1, col=1)
fig.add_trace(grey(img[z]), row=1, col=2)
fig.add_trace(overlay(predicted[z].astype(float), PRED_COLOUR, "predicted", 0.5), row=1, col=2)
if target_mask is not None:
    fig.add_trace(contour(target_mask[z], TARGET_COLOUR, "truth"), row=1, col=2)
heat = heatmap[0, 0].float().cpu().numpy()
hz = int(np.clip(round(z * (heat.shape[0] - 1) / max(img.shape[0] - 1, 1)), 0, heat.shape[0] - 1))
fig.add_trace(
    go.Heatmap(
        z=heat[hz], colorscale="Magma", showscale=True,
        colorbar=dict(len=0.85, thickness=12, x=1.005),
        hovertemplate="logit %{z:.2f}<extra></extra>",
    ),
    row=1, col=3,
)
fig.update_layout(title="7 · outputs  ·  cyan = predicted, yellow = truth (never shown to the model)")
show(fig, stem="forward_7_outputs")

print("outputs")
print(f"  predicted voxels @ {threshold}  : {int(predicted.sum())}")
print(f"  null head                       : {float(valid):+.4f}  ({'valid' if float(valid) > 0 else 'empty'})")
print(f"  centroid (x,y,z) world          : {tuple(round(float(v), 2) for v in centroid[0])}")

if target_mask is not None:
    dice, iou = dice_iou(
        torch.from_numpy(probability)[None, None],
        torch.from_numpy(target_mask)[None, None],
        threshold,
    )
    error = float((centroid[0].cpu() - truth_c[0]).norm()) if truth_c is not None else float("nan")
    print(f"  target                          : {target_name}")
    print(f"  true voxels                     : {int(target_mask.sum())}")
    print(f"  Dice / IoU                      : {float(dice.reshape(-1)[0]):.4f} / {float(iou.reshape(-1)[0]):.4f}")
    print(f"  centroid error                  : {error:.2f} world units")
    print("  (illustrative: one example, one seed. A reportable Dice carries the prompt-blind")
    print("   floor, the counterfactuals and the gate — CLAUDE.md §6.)")

with torch.no_grad():
    packed = model(image, direction_ids, name_ids)
delta = float((packed.logits - logits).abs().max())
print(f"manual vs StageB.forward: max |Δ logits| = {delta:.3e}")


  -> vizualization/out/forward_7_outputs.html


outputs
  predicted voxels @ 0.5  : 1988
  null head                       : +3.1840  (valid)
  centroid (x,y,z) world          : (17.82, 51.01, 35.86)
  target                          : pyramid
  true voxels                     : 2005
  Dice / IoU                      : 0.9847 / 0.9699
  centroid error                  : 2.98 world units
  (illustrative: one example, one seed. A reportable Dice carries the prompt-blind
   floor, the counterfactuals and the gate — CLAUDE.md §6.)
manual vs StageB.forward: max |Δ logits| = 1.308e-03
